# Website Traffic Prediction Using Apache Spark and Machine Learning

**End-to-End Big Data Analytics & Time-Series Regression**

This notebook demonstrates the end-to-end data pipeline using **Apache Spark (PySpark)** and **Spark MLlib** for website traffic (page views) forecasting.

### Workflow Stages:
1. SparkSession Initialization and Environment Configuration
2. Raw Dataset Ingestion & Schema Exploration
3. Spark Data Cleaning & Type Preprocessing
4. Exploratory Data Analysis (EDA) & Traffic Seasonality
5. Feature Engineering: Calendar, Lag, and Rolling Averages (Strictly Preventing Leakage)
6. Spark MLlib Machine Learning Pipeline (StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler)
7. Model Training: Linear Regression vs Random Forest Regressor
8. Model Evaluation: RMSE, MAE, R-Squared Metrics
9. Predictions & Residual Diagnostics

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add root to sys.path
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from config import config
from src.spark_session import get_spark_session, stop_spark_session
from pyspark.sql import functions as F, Window

print(f'Python Version: {sys.version}')

## 1. Initialize Apache Spark Session

In [2]:
spark = get_spark_session(app_name='WebsiteTrafficEDA')
print('Spark Version:', spark.version)

## 2. Ingest Raw Dataset & Inspect Schema

In [3]:
raw_df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(str(config.RAW_DATA_PATH))
print('Raw Schema:')
raw_df.printSchema()
print('Total Records:', raw_df.count())
raw_df.show(5)

## 3. Data Cleaning and Type Standardization

In [4]:
from src.preprocessing import preprocess_data
cleaned_df = preprocess_data(raw_df)
print('Cleaned Schema:')
cleaned_df.printSchema()
cleaned_df.show(5)

## 4. Exploratory Data Analysis & Seasonality Visualization

In [5]:
pdf = cleaned_df.toPandas()
pdf['Date'] = pd.to_datetime(pdf['Date'])

plt.figure(figsize=(14, 5))
plt.plot(pdf['Date'], pdf['Page_Loads'], label='Daily Page Views', color='#2563EB', alpha=0.7)
plt.plot(pdf['Date'], pdf['Page_Loads'].rolling(30).mean(), label='30-Day Moving Average', color='#DC2626', linewidth=2)
plt.title('Historical Website Traffic (2014 - 2020)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Page Loads')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 5. Feature Engineering: Calendar, Lag, and Rolling Averages

In [6]:
from src.feature_engineering import generate_features
features_df = generate_features(cleaned_df)
print('Engineered Columns:', features_df.columns)
features_df.select('Date', 'Day', 'Page_Loads', 'Lag_1_Page_Loads', 'Lag_7_Page_Loads', 'Rolling_7_Page_Loads').show(5)

## 6. Spark MLlib Pipelines & Model Training

In [7]:
from src.model_training import create_chronological_split, train_models
train_df, test_df = create_chronological_split(features_df, train_ratio=0.8)

artifacts = train_models(train_df, test_df)
lr_model = artifacts['lr_model']
rf_model = artifacts['rf_model']
train_transformed = artifacts['train_transformed']
test_transformed = artifacts['test_transformed']
print('MLlib Pipeline and Models Trained Successfully!')

## 7. Model Evaluation: RMSE, MAE, R-Squared

In [8]:
from src.model_evaluation import evaluate_models
metrics_df, best_model, detailed = evaluate_models(lr_model, rf_model, train_transformed, test_transformed)
print(metrics_df)

## 8. Generate Predictions and Error Analysis

In [9]:
from src.prediction import generate_batch_predictions
preds_df = generate_batch_predictions(rf_model, test_transformed, config.PREDICTIONS_PATH)
print('Sample Predictions:')
print(preds_df.head(10))

# Stop Spark Session
stop_spark_session(spark)